# GOV-01 V2-E0/E1: Seven-Class Majority Baseline and Compact CNN

Run this notebook one cell at a time in Google Colab. It uses only V2 `train` and `validation` images. It must not load or evaluate `protected_test`.

Before starting, create `multiclass_final.zip` from the local materialized V2 folder, upload that ZIP through the Colab Files panel, and run the setup cell below. This follows the same upload-and-extract approach used for V1. The folder contains `train`, `validation`, and `protected_test`; this notebook deliberately opens only the first two.

In [ ]:
from pathlib import Path
import zipfile

ARCHIVE_PATH = Path('/content/multiclass_final.zip')
DATA_DIR = Path('/content/data/processed/v2/multiclass_final')
OUTPUT_DIR = Path('/content/v2_e0_e1_output')

if not DATA_DIR.is_dir():
    if not ARCHIVE_PATH.is_file():
        print('ACTION REQUIRED: Upload multiclass_final.zip with the Colab Files panel, then rerun this cell.')
    else:
        with zipfile.ZipFile(ARCHIVE_PATH) as archive:
            archive.extractall(DATA_DIR.parent)
        print('Extracted:', DATA_DIR)

if DATA_DIR.is_dir():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print('Training data:', DATA_DIR)
    print('Output folder:', OUTPUT_DIR)
    print('The protected-test folder is deliberately not loaded in this notebook.')

In [ ]:
import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_NAMES = ['crack', 'manhole_cover', 'normal_asphalt', 'pothole', 'repaired_road', 'speed_bump', 'unpaved_road']

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print('TensorFlow:', tf.__version__)
print('Visible GPUs:', tf.config.list_physical_devices('GPU'))
print('Class order:', CLASS_NAMES)

In [ ]:
def count_images(split_name):
    result = {}
    for class_name in CLASS_NAMES:
        class_dir = DATA_DIR / split_name / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f'Missing class folder: {class_dir}')
        result[class_name] = len([path for path in class_dir.iterdir() if path.is_file()])
    return result

train_counts = count_images('train')
validation_counts = count_images('validation')
print('Train:', train_counts, 'total=', sum(train_counts.values()))
print('Validation:', validation_counts, 'total=', sum(validation_counts.values()))
assert sum(train_counts.values()) == 9452
assert sum(validation_counts.values()) == 1658

In [ ]:
# V2-E0: no neural-network training. Predict the most common training class for every validation image.
majority_class = max(train_counts, key=train_counts.get)
y_validation = np.concatenate([np.full(count, CLASS_NAMES.index(label)) for label, count in validation_counts.items()])
majority_predictions = np.full_like(y_validation, CLASS_NAMES.index(majority_class))
e0_metrics = {
    'run_name': 'v2_e0_majority_class',
    'majority_class': majority_class,
    'validation_accuracy': float(accuracy_score(y_validation, majority_predictions)),
    'validation_macro_f1': float(f1_score(y_validation, majority_predictions, average='macro', zero_division=0)),
}
print(json.dumps(e0_metrics, indent=2))
(OUTPUT_DIR / 'v2_e0_metrics.json').write_text(json.dumps(e0_metrics, indent=2) + '\n')

In [ ]:
# These are the only two datasets loaded. Validation has no random augmentation.
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'train', class_names=CLASS_NAMES, label_mode='int', image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE, shuffle=True, seed=SEED,
)
validation_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'validation', class_names=CLASS_NAMES, label_mode='int', image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE, shuffle=False,
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
validation_ds = validation_ds.prefetch(AUTOTUNE)

In [ ]:
# V2-E1: compact, unweighted CNN baseline.
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=IMAGE_SIZE + (3,)),
    tf.keras.layers.Rescaling(1.0 / 255),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax', name='road_condition'),
], name='v2_e1_compact_cnn')
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])
model.summary()

In [ ]:
checkpoint_path = OUTPUT_DIR / 'v2_e1_compact_cnn_best.keras'
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True),
]

start_time = time.time()
history = model.fit(train_ds, validation_data=validation_ds, epochs=20, callbacks=callbacks, verbose=1)
training_seconds = time.time() - start_time
print(f'Training time: {training_seconds:.1f} seconds')

In [ ]:
y_validation = np.concatenate([labels.numpy() for _, labels in validation_ds])
probabilities = model.predict(validation_ds, verbose=0)
predictions = probabilities.argmax(axis=1)

e1_metrics = {
    'run_name': 'v2_e1_compact_cnn',
    'validation_accuracy': float(accuracy_score(y_validation, predictions)),
    'validation_macro_f1': float(f1_score(y_validation, predictions, average='macro', zero_division=0)),
    'epochs_completed': len(history.history['loss']),
    'training_seconds': training_seconds,
    'class_weight': 'none',
    'classification_report': classification_report(y_validation, predictions, target_names=CLASS_NAMES, output_dict=True, zero_division=0),
}
(OUTPUT_DIR / 'v2_e1_validation_metrics.json').write_text(json.dumps(e1_metrics, indent=2) + '\n')
print(json.dumps({key: value for key, value in e1_metrics.items() if key != 'classification_report'}, indent=2))

ConfusionMatrixDisplay.from_predictions(y_validation, predictions, display_labels=CLASS_NAMES, xticks_rotation=45)
plt.title('V2-E1 validation confusion matrix')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'v2_e1_validation_confusion_matrix.png', dpi=150)
plt.show()

plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='validation loss')
plt.title('V2-E1 compact CNN loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'v2_e1_learning_curve.png', dpi=150)
plt.show()

print('V2-E1 evidence saved to Google Drive. Do not load protected_test.')